# SAP News Collection (RSS Feed)

Fetches the latest articles from the official SAP News RSS feed, scrapes the full article text for each entry with `newspaper3k`, and saves the result to `data/sap_news.json` for later merging.

In [ ]:
from pathlib import Path

import feedparser
import pandas as pd
from newspaper import Article

## Fetch feed entries

In [ ]:
FEED_URL = "https://news.sap.com/feed/"

feed = feedparser.parse(FEED_URL)
print(f"Fetched {len(feed.entries)} entries from {FEED_URL}")

In [ ]:
sap_docs = [
    {
        "title": entry.title,
        "url": entry.link,
        "published": entry.published,
        "source": "SAP News",
        "description": "",  # RSS feed has no separate description field
    }
    for entry in feed.entries
]

sap_df = pd.DataFrame(sap_docs)
sap_df.head()

## Scrape full article content

Downloads and parses each article with `newspaper3k`. A failed download (network error, paywall, etc.) falls back to an empty string instead of crashing the whole loop.

In [ ]:
contents = []

for url in sap_df["url"]:
    try:
        article = Article(url)
        article.download()
        article.parse()
        contents.append(article.text)
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
        contents.append("")

sap_df["content"] = contents

## Save to JSON

In [ ]:
DATA_DIR = Path("..") / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = DATA_DIR / "sap_news.json"
sap_df.to_json(output_path, orient="records", indent=2)

print(f"Saved {len(sap_df)} articles to {output_path}")